# Can we predict 12-month loan default?
Target: `defaulted`. Methodology follows `checklist.md`.

**Data facts that drive the design**
- 8000 rows, one row per `application_id` (unique), so no entity appears in several rows -> no grouping needed.
- There is no date column. File order is the only possible time proxy, so I treat it as chronological: the **last 20% of rows is the test set**, and CV inside the training rows uses forward-chaining (earlier rows train, later rows validate). If the file order is not chronological this is simply a non-random holdout, which is still valid.
- All feature columns are recorded at application, i.e. before funding, so all are known at prediction time. `application_id` is an identifier and is dropped.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

df = pd.read_csv('data.csv')
assert df.application_id.is_unique
TARGET = 'defaulted'
NUM = ['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
CAT = ['region']
FEATURES = NUM + CAT   # application_id dropped (identifier)
print(df.shape, df[TARGET].mean())

## 1. Set aside the test set (latest 20% of rows) before any modelling

In [ ]:
n_test = int(len(df)*0.2)
train, test = df.iloc[:-n_test].copy(), df.iloc[-n_test:].copy()
X_tr, y_tr = train[FEATURES], train[TARGET]
X_te, y_te = test[FEATURES], test[TARGET]
print(len(train), len(test), y_tr.mean(), y_te.mean())
# the test set is not touched again until the final scoring cell

## 4. Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial; AUC 0.5 by construction).
2. `logreg`: standardised numerics + one-hot region, L2 logistic regression; grid over `C` in {0.01, 0.1, 1, 10}.
3. `hgb`: histogram gradient boosting; grid over `max_depth` in {2, 3}, `learning_rate` in {0.03, 0.1}, `max_iter` in {100, 300}, with `min_samples_leaf=50`.

Selection metric: mean ROC AUC across CV folds (threshold-free). Decision threshold: chosen on pooled out-of-fold predictions to maximise F1 (default is rare, ~10%, so 0.5 is meaningless). Average precision is reported as a secondary metric. All preprocessing lives inside the Pipeline, so it is refit within each fold (checklist 3).

In [ ]:
def make(name, **p):
    pre = ColumnTransformer([('num', StandardScaler(), NUM), ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    if name == 'baseline': m = DummyClassifier(strategy='prior')
    elif name == 'logreg': m = LogisticRegression(C=p['C'], max_iter=2000)
    else: m = HistGradientBoostingClassifier(min_samples_leaf=50, random_state=0, **p)
    return Pipeline([('pre', pre), ('m', m)])

CANDIDATES = [('baseline', {})] \
  + [('logreg', p) for p in ParameterGrid({'C': [0.01, 0.1, 1, 10]})] \
  + [('hgb', p) for p in ParameterGrid({'max_depth': [2, 3], 'learning_rate': [0.03, 0.1], 'max_iter': [100, 300]})]
len(CANDIDATES)

## 2-3. Forward-chaining cross-validation on training rows only

In [ ]:
cv = TimeSeriesSplit(n_splits=5)
rows, oof = [], {}
for i, (name, p) in enumerate(CANDIDATES):
    pred = np.full(len(train), np.nan); aucs = []; aps = []
    for tr, va in cv.split(X_tr):
        pipe = make(name, **p).fit(X_tr.iloc[tr], y_tr.iloc[tr])
        s = pipe.predict_proba(X_tr.iloc[va])[:, 1]
        pred[va] = s; aucs.append(roc_auc_score(y_tr.iloc[va], s)); aps.append(average_precision_score(y_tr.iloc[va], s))
    oof[i] = pred
    rows.append(dict(idx=i, model=name, params=p, auc=np.mean(aucs), auc_sd=np.std(aucs), ap=np.mean(aps)))
res = pd.DataFrame(rows).sort_values('auc', ascending=False)
res.head(8)

## 5. Choose model and threshold on CV only

In [ ]:
best = res.iloc[0]
print('baseline CV AUC:', res[res.model == 'baseline'].auc.iloc[0])
print('chosen:', best.model, best.params, 'CV AUC %.3f +- %.3f' % (best.auc, best.auc_sd))
mask = ~np.isnan(oof[best.idx])          # rows that were ever in a validation fold
s, y = oof[best.idx][mask], y_tr.values[mask]
grid = np.linspace(0.02, 0.6, 59)
f1s = [f1_score(y, s >= t, zero_division=0) for t in grid]
THRESH = float(grid[int(np.argmax(f1s))])
print('threshold %.2f, OOF F1 %.3f, precision %.3f, recall %.3f' % (THRESH, max(f1s), precision_score(y, s >= THRESH), recall_score(y, s >= THRESH)))

**Decision recorded before touching the test set:** the model, hyperparameters and threshold printed above are final. The test set is scored exactly once below, with no further changes.

## 6. Refit on all training rows and score the test set once

In [ ]:
final = make(best.model, **best.params).fit(X_tr, y_tr)
p_te = final.predict_proba(X_te)[:, 1]
base = make('baseline').fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
pred = p_te >= THRESH
print('TEST ROC AUC  %.3f   (baseline %.3f)' % (roc_auc_score(y_te, p_te), roc_auc_score(y_te, base)))
print('TEST avg prec %.3f   (prevalence %.3f)' % (average_precision_score(y_te, p_te), y_te.mean()))
print('TEST at threshold %.2f: precision %.3f recall %.3f F1 %.3f' % (THRESH, precision_score(y_te, pred), recall_score(y_te, pred), f1_score(y_te, pred)))
# bootstrap CI for the headline AUC (fixed model, resampling test rows)
rng = np.random.default_rng(0); yv = y_te.values
bs = [roc_auc_score(yv[i], p_te[i]) for i in (rng.integers(0, len(yv), len(yv)) for _ in range(1000))]
print('AUC 95%% CI: %.3f - %.3f' % tuple(np.percentile(bs, [2.5, 97.5])))

## Conclusion
See the test-set cell above for the headline numbers (single, final evaluation). Read them against the baseline (AUC 0.5) and prevalence (~10%): the ROC AUC and its bootstrap interval say how much signal the application-time features carry, and the precision/recall at the CV-chosen threshold say what that means operationally.

Caveats: no timestamps exist, so "latest rows" relies on file order; the CI reflects test-sample noise only, not model-selection variance; no cost matrix was provided, so F1 is a stand-in for the true threshold criterion.